# Block 2 · the data, settled

Everything the model weeks start from, done once and shown in the open.

By the end you will have seen every decision that `hotels.py` makes for you,
including the one that would have given us a perfect score and a worthless
model. The last section leaves four questions open. Those are yours.

**Before you run this:** from the root of the labs repository,

```
python data/download.py --hotels
```

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import hotels

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

## 1 · What one row is

One booking, made at one moment, for one future stay at one of two Portuguese
hotels. `load_bookings()` gives the published file untouched, plus a few
columns every lab needs: the arrival date, the **booking date** (arrival minus
the lead time, which is when the booking was actually made), and the totals.

In [ ]:
raw = hotels.load_bookings()
print(f"{len(raw):,} bookings, {raw.shape[1]} columns")
print(f"arrivals {raw['arrival_date'].min().date()} to {raw['arrival_date'].max().date()}")
print(f"booked   {raw['booking_date'].min().date()} to {raw['booking_date'].max().date()}")
raw[["hotel", "lead_time", "arrival_date", "booking_date", "total_nights",
     "total_guests", "adr", "is_canceled"]].head()

Bookings go back to June 2013 even though the first arrival is July 2015,
because some lead times are over two years. That is real, not a bug.

## 2 · The two questions

Half the room predicts the nightly rate, half predicts cancellation. Same
table, same rows, different column in the `target` slot.

In [ ]:
print(f"cancelled:  {raw['is_canceled'].mean():.1%} of all bookings")
print()
print("adr, the nightly rate in euros:")
print(raw["adr"].describe()[["min", "25%", "50%", "75%", "max"]].round(2).to_string())
print()
print(f"exactly zero: {(raw['adr'] == 0).sum():,} bookings")

A minimum below zero and a maximum of 5,400 in the same column. Hold that
thought for section 4.

## 3 · What exists at the moment of booking

This is the question Week 1 taught you to ask first, and this dataset answers
it badly if you do not ask.

Five columns describe what happened *after* the booking was made. One of them
is the target wearing a different hat.

In [ ]:
print(raw.groupby(["reservation_status", "is_canceled"]).size().to_string())

`reservation_status` is *Canceled* exactly when `is_canceled` is 1. They are
the same fact written twice.

Let us use it anyway, once, so you know what a leak looks like from the inside.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


def tree_pipeline(numeric, categorical, model):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), numeric),
            ("cat", Pipeline([
                ("fill", SimpleImputer(strategy="constant", fill_value="none")),
                ("code", OrdinalEncoder(handle_unknown="use_encoded_value",
                                        unknown_value=-1, min_frequency=30)),
            ]), categorical),
        ])),
        ("model", model),
    ])


def score(pipe, train, valid, numeric, categorical):
    pipe.fit(train[numeric + categorical], train["is_canceled"])
    s = pipe.predict_proba(valid[numeric + categorical])[:, 1]
    ap = average_precision_score(valid["is_canceled"], s)
    hits = pipe.predict(valid[numeric + categorical]) == valid["is_canceled"]
    return {"PR AUC": round(float(ap), 3),
            "lift": round(float(ap / valid["is_canceled"].mean()), 2),
            "ROC AUC": round(float(roc_auc_score(valid["is_canceled"], s)), 3),
            "accuracy": round(float(hits.mean()), 3)}


df = hotels.split_by_booking_date(hotels.clean_bookings(raw))
train, valid, test = hotels.splits(df)

numeric = hotels.AT_BOOKING_NUMERIC
categorical = hotels.AT_BOOKING_CATEGORICAL
mask = [False] * len(numeric) + [True] * len(categorical)

leaky = tree_pipeline(numeric, categorical + ["reservation_status"],
                      HistGradientBoostingClassifier(
                          random_state=42,
                          categorical_features=mask + [True]))
print("with reservation_status:   ",
      score(leaky, train, valid, numeric, categorical + ["reservation_status"]))

In [ ]:
honest = tree_pipeline(numeric, categorical, HistGradientBoostingClassifier(
    random_state=42, categorical_features=mask))
print("without reservation_status:", score(honest, train, valid, numeric, categorical))

Three scores of 1.000, no error, no warning. Nothing in the output tells you
anything is wrong — you have to know to look.

`hotels.py` removes five columns from both questions for this reason:
`reservation_status` and `reservation_status_date` because they are the answer,
`assigned_room_type` because the room is assigned at check-in,
`booking_changes` and `days_in_waiting_list` because they accumulate after the
booking is made.

In [ ]:
print(hotels.NOT_KNOWN_AT_BOOKING)

## 4 · What is wrong in here

Six checks, the same list Week 2 used: missing, impossible, inconsistent,
duplicated, out of range, misnamed.

In [ ]:
guests = raw[["adults", "children", "babies"]].fillna(0).sum(axis=1)
nights = raw["stays_in_weekend_nights"] + raw["stays_in_week_nights"]

checks = {
    "nobody in the booking": guests == 0,
    "no nights booked": nights == 0,
    "a negative nightly rate": raw["adr"] < 0,
    "a rate above 1,000": raw["adr"] > 1000,
    "a rate of exactly zero": raw["adr"] == 0,
    "children missing": raw["children"].isna(),
    "meal recorded as 'Undefined'": raw["meal"] == "Undefined",
    "market segment 'Undefined'": raw["market_segment"] == "Undefined",
}
pd.Series({name: int(m.sum()) for name, m in checks.items()},
          name="bookings").to_frame()

Only three of those are removed, and the test is whether the row could be real
at all. A booking with nobody in it, or no nights in it, or a price below zero,
is a recording error whichever question you are answering.

In [ ]:
clean, log = hotels.clean_bookings(raw, log=True)
print(log.to_string())
print(f"\n{len(clean):,} bookings left")

Everything else stays, including things that look wrong:

- **1,959 bookings at exactly zero.** Complimentary stays and some group
  allocations. Real bookings, so cancellation keeps them; the rate question
  leaves them out, because *what will this pay* is not *will this be free*.
- **One booking at €5,400.** Possibly real, possibly a typo. `daily_rate()`
  keeps €1 to €1,000.
- **Four missing `children`.** Four. The imputer handles it.

One recording habit is fixed: `meal` writes self-catering as both `SC` and
`Undefined`, so `Undefined` becomes `SC`.

## 5 · The split

Split on the **booking date**, not the arrival date. At the moment we predict,
the arrival has not happened.

In [ ]:
df = hotels.split_by_booking_date(clean)
summary = (df.groupby("split", observed=True)
             .agg(bookings=("is_canceled", "size"),
                  cancelled=("is_canceled", "mean"),
                  first_booked=("booking_date", "min"),
                  last_booked=("booking_date", "max"),
                  first_arrival=("arrival_date", "min"),
                  last_arrival=("arrival_date", "max"))
             .reindex(["train", "valid", "test"]))
summary["cancelled"] = (summary["cancelled"] * 100).round(1)
summary

Look at the arrival columns. They **overlap** across the splits, while the
booking columns do not. That is correct: a booking made in September 2016 for
next summer belongs in training, because September 2016 is when we would have
known about it.

Splitting on arrival instead — the obvious thing to do — would train on
bookings that were made after some of the bookings being predicted.

And notice the cancellation rate: 38.6%, 39.9%, 31.8%. The world moves.

What does the honest split cost?

In [ ]:
from sklearn.model_selection import train_test_split

rtrain, rvalid = train_test_split(df, test_size=0.2, random_state=42)
random_split = tree_pipeline(numeric, categorical, HistGradientBoostingClassifier(
    random_state=42, categorical_features=mask))
print("random split:         ", score(random_split, rtrain, rvalid, numeric, categorical))

train, valid, test = hotels.splits(df)
print("split by booking date:", score(honest, train, valid, numeric, categorical))

## 6 · The categories

Three columns are wide. One is nearly empty.

In [ ]:
rows = []
for column in ("country", "agent", "company", "market_segment", "deposit_type"):
    share = df[column].value_counts(normalize=True)
    rows.append({"column": column,
                 "values": df[column].nunique(),
                 "commonest": share.index[0],
                 "its share": f"{share.iloc[0]:.1%}",
                 "values for 95%": int(np.searchsorted(share.cumsum(), 0.95) + 1)})
pd.DataFrame(rows).set_index("column")

`company` needs two values to cover 95% of bookings, because 94% of them have
no company at all. Whether that column is worth keeping is a decision, not a
fact — section 9.

Before moving on, one finding worth arguing about.

In [ ]:
by_deposit = (df.groupby("deposit_type")["is_canceled"]
                .agg(bookings="size", cancelled="mean")
                .sort_values("cancelled"))
by_deposit["cancelled"] = (by_deposit["cancelled"] * 100).round(1)
by_deposit

Non-refundable bookings are cancelled **99.4%** of the time, across 14,587 of
them. That is not a leak — the deposit type is agreed at booking — but it is
the opposite of what a deposit is for. Something about how this hotel records
non-refundable bookings is not what the column name suggests, and the data
cannot tell you what. Use it, and write down that you do not know why.

## 7 · The floor

Both tracks, from the training rows only.

In [ ]:
cancel = hotels.cancellation(df)
rate = hotels.daily_rate(df)

ctrain, cvalid, _ = hotels.splits(cancel)
rtrain_, rvalid_, _ = hotels.splits(rate)

print("track B, cancellation:", {k: round(v, 3) for k, v
                                 in hotels.baseline(ctrain, cvalid, "cancellation").items()})
print("track A, nightly rate:", {k: round(v, 2) for k, v
                                 in hotels.baseline(rtrain_, rvalid_, "rate").items()})
print(f"\ncancellation rows: {len(cancel):,}   rate rows: {len(rate):,}")

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def linear_pipeline(model):
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                              ("s", StandardScaler())]), numeric),
            ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="none")),
                              ("o", OneHotEncoder(handle_unknown="ignore"))]), categorical),
        ])),
        ("model", model),
    ])


logistic = linear_pipeline(LogisticRegression(max_iter=2000))
logistic.fit(hotels.features(ctrain), ctrain["target"])
s = logistic.predict_proba(hotels.features(cvalid))[:, 1]
ap = average_precision_score(cvalid["target"], s)
print(f"logistic:  PR AUC {ap:.3f}   lift {ap / cvalid['target'].mean():.2f}")

linear = linear_pipeline(LinearRegression())
linear.fit(hotels.features(rtrain_), rtrain_["target"])
p = linear.predict(hotels.features(rvalid_))
print(f"linear:    R2 {r2_score(rvalid_['target'], p):.3f}   "
      f"MAE EUR {mean_absolute_error(rvalid_['target'], p):.2f}")

Those two lines are what Week 5 starts from and what every later week has to
beat.

## 8 · One line, from here on

Everything above is what `block_2()` does. From Week 5 your lab starts like
this and nothing else:

In [ ]:
df = hotels.block_2()               # or block_2(sample=20_000) while developing
cancel = hotels.cancellation(df)
rate = hotels.daily_rate(df)

print(df.groupby("split", observed=True).size().reindex(["train", "valid", "test"]).to_string())

## 9 · Your turn

Four questions this notebook deliberately did not answer. There is no checker
for these: they are judgement calls, and Project 2 will ask you to defend the
ones you make.

**a. How should the wide columns be encoded?** One-hot on `country` costs 177
columns, of which 154 cover 5% of the data between them. Group the rare ones
into "other"? Target-encode, cross-fitted, the way Week 3 did? Try one, measure
it on validation, and write down what it cost.

In [ ]:
# TODO (a): pick one encoding for `country` and compare it against plain
# one-hot on validation. Keep everything else the same.

**b. Is `company` worth keeping at all?** 94% of bookings have none. Does
dropping it change anything? Does replacing it with a single yes/no "booked
through a company" column do better than the 348 values?

In [ ]:
# TODO (b): measure all three — keep it, drop it, reduce it to yes/no.

**c. What is worth deriving?** `hotels.py` gives you `total_nights`,
`total_guests` and numeric date parts. A ratio of previous cancellations to
previous stays, a weekend-arrival flag, lead-time buckets, a "has special
requests" flag. Build two or three and see whether they earn their place.

In [ ]:
# TODO (c): build two derived features and test whether they help.

**d. What about the rows cleaning left in?** The 1,959 zero-rate bookings, the
€5,400 outlier, the `Undefined` segments. Decide what you would do for your
track, and say why in one sentence each.

In [ ]:
# TODO (d): your decision, and your reason.

---

Next: [Week 5 · Fit](https://evisp.github.io/ml-course-umt/03-models-optimization/week-05-linear-models/).